# pré-traitement (réduction) des données capturées
 $\rightarrow$ **générer les DOF (dark, offset flat) pour corriger les images de sciences des défauts du chemin optique, de la caméra et de l'atmosphère terrestre**



## définition des répertoires et fichiers

In [7]:
# répertoire de travail
#CAPTURE_DIR='data/20250807_deneb_ruchbah_gamcas/'
#CAPTURE_DIR='../../../CAPTURES/20250822_v1296Aql_altair_Tcrb_rsOph/'
#CAPTURE_DIR='../../../CAPTURES/20251102_cxdra_v1770cyg_deneb/'
CAPTURE_DIR = '../../../CAPTURES/20260224_m97_merak_procyon/'


# bruts sciences
#CAPTURE_SCIENCE = "V1295Aql-300s-*.fits"
#CAPTURE_SCIENCE = "gamcas-5s-*.fit"
#CAPTURE_SCIENCE = "ruchbah-5s-*.fit"
#CAPTURE_SCIENCE = "lamCyg-60s-*.fits"
#CAPTURE_SCIENCE = "V1770Cyg-60s-*.fits"
CAPTURE_SCIENCE = "merak-1s-*.fits"
#CAPTURE_SCIENCE = "m97-600s-*.fits"
#CAPTURE_SCIENCE = "sun-*.fits"

# bruts offset
CAPTURE_BIAS = "Bias-0s-*.fits"

# bruts dark
#CAPTURE_DARK = "Dark-5s-*.fit"
#CAPTURE_DARK = "dark10m_DD200-0*.fit"
CAPTURE_DARK = "Dark-5s-*.fits"

# bruts flat
#CAPTURE_FLAT = "flat_tung_-1s*.fit"
#CAPTURE_FLAT = "flat-0-5s-*.fits"
CAPTURE_FLAT = "flat_table-0-5s-210-*.fits"

# bruts néon
#CAPTURE_CALIB = "neon-30s-*.fit"s
#CAPTURE_CALIB = "neon2-5s-*.fits"
CAPTURE_CALIB = "neon_table-5s-210-*.fits"

# crop (vertical Y px) sinon None pour pas de crop 
TRIM_TOP = 700 #None 
TRIM_BOTTOM = 1250 #None

# fichiers de sortie
OUTPUT_BIAS = 'master_bias.fits'
OUTPUT_DARK = 'master_dark.fits'
OUTPUT_FLAT = 'master_flat.fits'
OUTPUT_CALIB = 'master_calib.fits'
OUTPUT_SCIENCE = 'master_science.fits'

import warnings, pathlib, os

# on affiche les fichiers 
for _type in (CAPTURE_BIAS, CAPTURE_DARK, CAPTURE_FLAT, CAPTURE_CALIB, CAPTURE_SCIENCE):
    print("\n",[f.name for f in pathlib.Path(CAPTURE_DIR).glob(_type)])



 ['Bias-0s-4.fits', 'Bias-0s-8.fits', 'Bias-0s-9.fits', 'Bias-0s-5.fits', 'Bias-0s-10.fits', 'Bias-0s-2.fits', 'Bias-0s-3.fits', 'Bias-0s-11.fits', 'Bias-0s-12.fits', 'Bias-0s-1.fits', 'Bias-0s-13.fits', 'Bias-0s-6.fits', 'Bias-0s-14.fits', 'Bias-0s-15.fits', 'Bias-0s-7.fits']

 ['Dark-5s-11.fits', 'Dark-5s-3.fits', 'Dark-5s-2.fits', 'Dark-5s-10.fits', 'Dark-5s-5.fits', 'Dark-5s-9.fits', 'Dark-5s-8.fits', 'Dark-5s-4.fits', 'Dark-5s-7.fits', 'Dark-5s-15.fits', 'Dark-5s-14.fits', 'Dark-5s-6.fits', 'Dark-5s-1.fits', 'Dark-5s-13.fits', 'Dark-5s-12.fits']

 ['flat_table-0-5s-210-1.fits', 'flat_table-0-5s-210-11.fits', 'flat_table-0-5s-210-6.fits', 'flat_table-0-5s-210-7.fits', 'flat_table-0-5s-210-10.fits', 'flat_table-0-5s-210-13.fits', 'flat_table-0-5s-210-8.fits', 'flat_table-0-5s-210-4.fits', 'flat_table-0-5s-210-5.fits', 'flat_table-0-5s-210-9.fits', 'flat_table-0-5s-210-12.fits', 'flat_table-0-5s-210-2.fits', 'flat_table-0-5s-210-15.fits', 'flat_table-0-5s-210-14.fits', 'flat_table-0

## création du dashboard
- lancer la cellule suivante
- souris sur '**Colormap**', bouton droit, menu "**create new view for cell output**"
- déplacer le nouvel onglet créé '**Output View**' pour le garder visible pendant que vous naviguez et exécutez les cellules de code

En cas de souci d'affichage $\rightarrow$  '**CTRL-R**'


In [2]:
%matplotlib widget
from image_widget import ImageWidget

db = ImageWidget()
db.show()


## import des libs 

In [3]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from astropy.nddata import CCDData
import ccdproc as ccdproc
import warnings, pathlib, os
from astropy.utils.exceptions import AstropyWarning, AstropyUserWarning
warnings.simplefilter('ignore', category=AstropyWarning)
warnings.simplefilter('ignore', category=AstropyUserWarning)
warnings.simplefilter('ignore', UserWarning)


# création du master bias

In [8]:
# charge toutes les images dans une liste de CCDData
CAPTURE_FILES = CAPTURE_BIAS
OUTPUT_FILE = OUTPUT_BIAS

img_list = [ccdproc.trim_image(CCDData.read(f, unit=u.Unit('adu'))[TRIM_TOP:TRIM_BOTTOM, :]) 
            for f in list(pathlib.Path(CAPTURE_DIR).glob(CAPTURE_FILES))]

# combine les images (sum, median, average, sigma_clip, trim, etc...)
img_combined = ccdproc.Combiner(img_list).median_combine()

# sauve l'image combinée
img_combined.write(CAPTURE_DIR + OUTPUT_FILE, overwrite=True) 

# affiche l'image
db.show_image(img_combined, name='master bias')


INFO: affichage de l'image master bias : bin=1, shape=(550, 3856), min=112, avg=651.3, max=848, stddev=7.9


# création du master dark

In [9]:
# nomme les images à combiner
CAPTURE_FILES = CAPTURE_DARK
OUTPUT_FILE = OUTPUT_DARK

# recharge l'offset combiné
master_bias = CCDData.read(CAPTURE_DIR + OUTPUT_BIAS, unit=u.Unit('adu'))

# charge toutes les images dans une liste de CCDData
img_list = [ccdproc.trim_image(CCDData.read(f, unit=u.Unit('adu'))[TRIM_TOP:TRIM_BOTTOM, :])
                               for f in list(pathlib.Path(CAPTURE_DIR).glob(CAPTURE_FILES))]

# récupére les temps d'intégration des dark qui sera utilisé plus tard
DARK_EXPTIME = img_list[0].meta['EXPTIME']

# supprime l'offset de chaque dark (car on va les 'scaler' plus tard -> il ne faut retirer l'offset)
img_corrected = [ccdproc.subtract_bias(ccd, master_bias) for ccd in img_list]

# combine les images (sum, median, average, sigma_clip, trim, etc...)
img_combined = ccdproc.Combiner(img_corrected).median_combine()

# mets à jour ses metadonnées à partir d'une image source
img_combined.meta = img_list[0].meta.copy()

# sauve l'image combinée
img_combined.write(CAPTURE_DIR + OUTPUT_FILE, overwrite=True) 

# affiche l'image
db.show_image(img_combined, name='master dark')



INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: affichage de l'image master dark : bin=1, shape=(550, 3856), min=-192, avg=-1.2, max=50736, stddev=44.3


# création du master flat

In [10]:
# nomme les images à combiner
CAPTURE_FILES = CAPTURE_FLAT
OUTPUT_FILE = OUTPUT_FLAT

# recharge l'offset combiné
master_bias = CCDData.read(CAPTURE_DIR + OUTPUT_BIAS, unit=u.Unit('adu'))

# charge toutes les images dans une liste de CCDData
img_list = [ccdproc.trim_image(CCDData.read(f, unit=u.Unit('adu'))[TRIM_TOP:TRIM_BOTTOM, :])
                               for f in list(pathlib.Path(CAPTURE_DIR).glob(CAPTURE_FILES))]

# supprime l'offset de chaque dark (car on va les 'scaler' plus tard -> il ne faut retirer l'offset)
img_corrected = [ccdproc.subtract_bias(ccd, master_bias) for ccd in img_list]

# combine les images (sum, median, average, sigma_clip, trim, etc...)
img_combined = ccdproc.Combiner(img_corrected).sum_combine() #median_combine()

# on remonte à > 0.0 et on normalise
min_val = np.min(img_combined)
epsilon = 1e-6  # pour éviter le zéro
img_combined = CCDData(img_combined.data - min_val + epsilon, unit=u.Unit('adu'))
img_combined = img_combined.divide(np.median(img_combined.data))

# mets à jour ses metadonnées à partir d'une image source
img_combined.meta = img_list[0].meta.copy()

# sauve l'image combinée
img_combined.write(CAPTURE_DIR + OUTPUT_FILE, overwrite=True) 

# affiche l'image
db.show_image(img_combined, name='master flat')



INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: affichage de l'image master flat : bin=1, shape=(550, 3856), min=0, avg=1.3, max=6, stddev=1.2


# création du master calib (néon)

In [11]:
# nomme les images à combiner
CAPTURE_FILES = CAPTURE_CALIB
OUTPUT_FILE = OUTPUT_CALIB

# recharge l'offset combiné
master_bias = CCDData.read(CAPTURE_DIR + OUTPUT_BIAS, unit=u.Unit('adu'))

# charge toutes les images dans une liste de CCDData
img_list = [ccdproc.trim_image(CCDData.read(f, unit=u.Unit('adu'))[TRIM_TOP:TRIM_BOTTOM, :])
            for f in list(pathlib.Path(CAPTURE_DIR).glob(CAPTURE_FILES))]

# supprime l'offset de chaque dark (car on va les 'scaler' plus tard -> il ne faut retirer l'offset)
img_corrected = [ccdproc.subtract_bias(ccd, master_bias) for ccd in img_list]

# combine les images (sum, median, average, sigma_clip, trim, etc...)
img_combined = ccdproc.Combiner(img_corrected).median_combine()

# mets à jour ses metadonnées à partir d'une image source
img_combined.meta = img_list[0].meta.copy()

# sauve l'image combinée
img_combined.write(CAPTURE_DIR + OUTPUT_FILE, overwrite=True) 

# affiche l'image
db.show_image(img_combined, name='master calib')


INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: affichage de l'image master calib : bin=1, shape=(550, 3856), min=-96, avg=5659.1, max=64944, stddev=15363.0


# création du master science

In [12]:
### fonction de recalage des spectres 2D
### astroalign ne fonctionne pas car pas d'étoiles
### --> on passe par une convolution FFT 

from scipy.signal import fftconvolve

"""
align a set of loaded frames - specific to spectra fields (fft based)
"""
def spec_align(img_list, ref_image_index: int = 0):    
    ### Collect arrays and crosscorrelate all (except the first) with the first.
    print('align: fftconvolve running...')
    nX, nY = img_list[ref_image_index].shape
    correlations = [fftconvolve(img_list[ref_image_index].data.astype('float32'),
                                image[::-1, ::-1].data.astype('float32'),
                                mode='same') 
                    for image in img_list[1:]]

    ### For each image determine the coordinate of maximum cross-correlation.
    print('align: get max cross-correlation for every image...')
    shift_indices = [np.unravel_index(np.argmax(corr_array, axis=None), corr_array.shape) 
                     for corr_array in correlations]
    
    deltas = [(ind[0] - int(nX / 2), ind[1] - int(nY / 2)) for ind in shift_indices]
    for dt in deltas: print(f"align: images deltas = {dt[0]:.0f} / {dt[1]:.0f}")  

    ### Warn for ghost images if realignment requires shifting by more than 15% of the field size.
    x_frac = abs(max(deltas, key=lambda x: abs(x[0]))[0]) / nX
    y_frac = abs(max(deltas, key=lambda x: abs(x[1]))[1]) / nY
    t_frac = max(x_frac, y_frac)
    if t_frac > 0.15:
        print('align warning : shifting by {}% of the field size'.format(int(100 * t_frac)))

    ### Roll the images to realign them and return their median.
    print('align: images realignement ...')
    realigned_images = [CCDData(np.roll(image, deltas[i], axis=(0, 1)), unit = u.adu, header = image.header) 
                        for (i, image) in enumerate(img_list[1:])]

    ### do not forget the reference image
    realigned_images.append(CCDData(img_list[ref_image_index].data.astype('float32'), unit = u.adu, header = img_list[ref_image_index].header))
    print('align: complete')
    
    return (realigned_images)


In [13]:
# nomme les images à combiner
CAPTURE_FILES = CAPTURE_SCIENCE
OUTPUT_FILE = OUTPUT_SCIENCE

# recharge les masters
master_bias = CCDData.read(CAPTURE_DIR + OUTPUT_BIAS, unit=u.Unit('adu'))
master_dark = CCDData.read(CAPTURE_DIR + OUTPUT_DARK, unit=u.Unit('adu'))
master_flat = CCDData.read(CAPTURE_DIR + OUTPUT_FLAT, unit=u.Unit('adu'))

# charge toutes les images dans une liste de CCDData
img_list = [ccdproc.trim_image(CCDData.read(f, unit=u.Unit('adu'))[TRIM_TOP:TRIM_BOTTOM, :])
            for f in list(pathlib.Path(CAPTURE_DIR).glob(CAPTURE_FILES))]

# on récupére le temps d'intégration des images sciences
SCIENCE_EXPTIME = img_list[0].meta['EXPTIME']
#print(f"{SCIENCE_EXPTIME=}, {DARK_EXPTIME=}")

# pour chaque image, on soustrait l'offset, scale le dark, soustrait le dark, divise le flat 
sciences_corr = []
for img in img_list:    
    science_corr = ccdproc.subtract_bias(ccd=img, master=master_bias) 
    science_corr = ccdproc.subtract_dark(ccd=science_corr, master=master_dark, 
                                         scale=True, 
                                         data_exposure=SCIENCE_EXPTIME* u.Unit('s'), 
                                         dark_exposure=DARK_EXPTIME* u.Unit('s'),
                                         exposure_unit=u.Unit('s'))
    science_corr = ccdproc.flat_correct(ccd=science_corr, flat=master_flat)
  
    sciences_corr.append(science_corr)


# optionel : on recale les images prétraitées par rapport à une référence (la 1ere)
#img_registered = spec_align(sciences_corr)
img_registered = sciences_corr

# on combine les images (sum, median, average, sigma_clip, trim, etc...)
img_combined = ccdproc.Combiner(img_registered).sum_combine()    #sum_combine() #median_combine() 

# on met à jour les metadonnées à partir d'une image source
img_combined.meta = img_list[0].meta.copy()

# et enfin on sauve l'image combinée
img_combined.write(CAPTURE_DIR + OUTPUT_FILE, overwrite=True) 

# on affiche l'image
db.show_image(img_combined, name='master science')


INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: affichage de l'image master science : bin=1, shape=(550, 3856), min=-216334780805611, avg=-102003087.4, max=1668776, stddev=148551276539.9


# correction des cosmics


In [14]:
# normalement pas nécessaire si combiner.median utilisé pour le master-science
# récupérer code de QuickSpec